In [ ]:

import sys
from pathlib import Path

# Retrouver le dossier principal du projet
racine = Path.cwd()
if racine.name == "notebooks":
    racine = racine.parent

# Permettre l'acces aux modules du projet
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

In [ ]:
from src import chargement

# Charger les ventes de Loire-Atlantique
ventes = chargement.charger_ventes("44")

# Construire l'echantillon d'analyse
echantillon = chargement.echantillon_analyse(ventes)

print("Ventes du département 44 :", len(ventes))
print("Ventes retenues pour l'analyse :", len(echantillon))

echantillon.head()

In [ ]:
# Construire un premier tableau descriptif par etiquette DPE
tableau_dpe = (
    echantillon.groupby("etiquette_dpe", observed=False)
    .agg(
        nombre_ventes=("id_mutation", "size"),
        prix_m2_moyen=("prix_m2", "mean"),
        prix_m2_median=("prix_m2", "median"),
        surface_mediane=("surface", "median"),
    )
    .round(2)
)

tableau_dpe

In [ ]:
# Comparer les etiquettes au sein de chaque type de logement
tableau_type_dpe = (
    echantillon.groupby(
        ["type_bien", "etiquette_dpe"],
        observed=False,
    )
    .agg(
        nombre_ventes=("id_mutation", "size"),
        prix_m2_median=("prix_m2", "median"),
        surface_mediane=("surface", "median"),
    )
    .round(2)
)

tableau_type_dpe

In [ ]:
import matplotlib.pyplot as plt

# Deux graphiques avec la même échelle verticale
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, type_bien in zip(axes, ["appartement", "maison"]):
    selection = echantillon[echantillon["type_bien"] == type_bien]

    selection.boxplot(
        column="prix_m2",
        by="etiquette_dpe",
        ax=ax,
        showfliers=False,
    )

    ax.set_title(type_bien.capitalize() + "s")
    ax.set_xlabel("Étiquette DPE")

axes[0].set_ylabel("Prix de vente (€/m²)")
axes[0].set_ylim(bottom=0)

fig.suptitle("Loire-Atlantique (44) — Prix au m² selon le DPE, 2022–2025")
fig.text(
    0.5, 0.02,
    "Points au-delà des moustaches masqués ; toutes les valeurs participent aux calculs.",
    ha="center", fontsize=9,
)
fig.tight_layout(rect=[0, 0.07, 1, 0.94])

# Enregistrer la figure pour le rapport
dossier_figures = racine / "reports" / "figures"
dossier_figures.mkdir(parents=True, exist_ok=True)
fig.savefig(dossier_figures / "boxplots_dpe_44_simple.png", dpi=200)

plt.show()

Dans l’échantillon de Loire-Atlantique étudié sur 2022–2025,
le prix médian au m² des maisons diminue globalement de la
classe A à la classe G, sans décroissance régulière entre
toutes les classes.

Pour les appartements, cette tendance est moins nette :
la classe G présente le prix médian au m² le plus élevé.
Ces appartements sont aussi plus petits en médiane :
31,86 m² contre 58,73 m² pour la classe A.

Les distributions de prix se recouvrent largement entre
classes DPE. Ces comparaisons sont descriptives : elles
ne permettent pas d’isoler l’effet du DPE, car la surface,
la localisation et les autres caractéristiques des biens
peuvent également expliquer les écarts observés.

In [ ]:
# Calculer la proportion de maisons dans chaque classe DPE
part_maisons = (
    echantillon.assign(est_maison=echantillon["type_bien"] == "maison")
    .groupby("etiquette_dpe", observed=False)["est_maison"]
    .mean()
    .mul(100)
)

# Ajouter cette information au tableau descriptif
tableau_dpe["part_maisons_pct"] = part_maisons.round(1)

tableau_dpe

In [ ]:
import pandas as pd

colonnes = ["periode_construction", "annee_construction"]

controle_construction = pd.DataFrame({
    "nombre_manquant": echantillon[colonnes].isna().sum(),
    "part_manquante_pct": (
        echantillon[colonnes].isna().mean().mul(100).round(1)
    ),
})

controle_construction

In [ ]:
# Placer les périodes dans l'ordre chronologique
ordre_periodes = [
    "avant 1948",
    "1948-1974",
    "1975-1977",
    "1978-1982",
    "1983-1988",
    "1989-2000",
    "2001-2005",
    "2006-2012",
    "2013-2021",
    "après 2021",
]

# Répartition des périodes à l'intérieur de chaque classe DPE
tableau_construction = (
    pd.crosstab(
        echantillon["periode_construction"],
        echantillon["etiquette_dpe"],
        normalize="columns",
    )
    .reindex(ordre_periodes)
    .mul(100)
)

tableau_construction.round(1)

Les périodes de construction déclarées diffèrent fortement
selon la classe DPE. Dans l’échantillon de Loire-Atlantique,
98,0 % des ventes classées G et 95,7 % de celles classées F
concernent des logements déclarés construits avant 1975,
contre 3,7 % des ventes classées A.

La période de construction constitue donc une caractéristique
à prendre en compte pour comparer les logements entre classes DPE.
Ces résultats décrivent une association et ne permettent pas
d’isoler l’effet de l’ancienneté sur la performance énergétique.

La description repose sur les périodes déclarées. Les désaccords
repérés entre certaines périodes et années de construction
constituent une limite de qualité des données.

In [ ]:
# Traduire les classes DPE en rangs numériques
rang_dpe = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

# Sélectionner les variables à comparer
donnees_correlation = echantillon[
    ["prix_m2", "surface", "conso_5_usages_par_m2_ep"]
].copy()

donnees_correlation["rang_dpe"] = (
    echantillon["etiquette_dpe"].map(rang_dpe).astype(float)
)

# Calculer les corrélations de Spearman
correlations = donnees_correlation.corr(method="spearman")

correlations.round(3)

In [ ]:
annee = pd.to_numeric(
    echantillon["annee_construction"], errors="coerce"
)
annee_vente = echantillon["date_mutation"].dt.year

# Limites des périodes définies dans la cellule précédente
bornes = [
    float("-inf"), 1947, 1974, 1977, 1982, 1988,
    2000, 2005, 2012, 2021, float("inf"),
]

periode_calculee = pd.cut(
    annee, bins=bornes, labels=ordre_periodes
)

coherence_periode = (
    periode_calculee.astype("string")
    .eq(echantillon["periode_construction"].astype("string"))
    .fillna(False)
)

print(
    "Années hors de la période déclarée :",
    (annee.notna() & ~coherence_periode).sum(),
)

In [ ]:
# Retenir les années cohérentes, positives et entières
age_valide = (
    coherence_periode
    & annee.gt(0)
    & annee.mod(1).eq(0)
    & annee.le(annee_vente)
)

donnees_correlation["age_bati"] = (
    annee_vente - annee
).where(age_valide)

print("Ancienneté exploitable :", age_valide.sum(), "ventes")

correlations = donnees_correlation.corr(method="spearman")

correlations.round(3)

In [ ]:
import matplotlib.pyplot as plt

libelles = [
    "Prix au m²", "Surface", "Consommation",
    "Rang DPE", "Ancienneté déclarée",
]

fig_corr, ax = plt.subplots(figsize=(8, 7))

# Fixer l'échelle des couleurs entre -1 et +1
image_corr = ax.imshow(
    correlations, cmap="RdBu_r", vmin=-1, vmax=1
)

# Inscrire chaque coefficient dans sa case
for i in range(len(correlations)):
    for j in range(len(correlations)):
        valeur = correlations.iloc[i, j]
        couleur = "white" if abs(valeur) >= 0.6 else "black"
        ax.text(
            j, i, f"{valeur:.3f}",
            ha="center", va="center",
            color=couleur, fontsize=11,
        )

ax.set_xticks(range(5), libelles, rotation=35, ha="right")
ax.set_yticks(range(5), libelles)
ax.set_title(
    "Corrélations de Spearman\nLoire-Atlantique (44), 2022–2025",
    pad=16,
)
fig_corr.colorbar(image_corr, ax=ax, label="Coefficient de Spearman")

# Préciser les effectifs utilisés
n_total = len(donnees_correlation)
n_age = donnees_correlation["age_bati"].notna().sum()

fig_corr.text(
    0.5, 0.02,
    f"Quatre premières variables : {n_total:,} ventes ; "
    f"avec l’ancienneté : {n_age:,} ventes.\n"
    "A = 1, G = 7. Ancienneté calculée à la vente sur les années cohérentes.\n"
    "Source : données DVF–DPE préparées du projet. Analyse descriptive.",
    ha="center", fontsize=9,
)

fig_corr.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

Dans les ventes disposant d’une année de construction exploitable,
l’ancienneté déclarée est positivement associée à la consommation
énergétique (Spearman : 0,626) et au rang DPE (0,616). Les logements
plus anciens tendent donc à présenter une consommation plus élevée
et un classement énergétique moins favorable.

Son association avec le prix au m² reste faible (-0,100).
Ces relations sont descriptives et ne contrôlent pas les différences
de localisation, de surface ou de type de bien.

Les corrélations impliquant l’ancienneté portent sur 30 758 ventes,
contre 47 492 pour les autres paires. Le sous-échantillon disposant
d’une ancienneté exploitable peut différer de l’échantillon complet.

In [ ]:
dossier_figures = racine / "reports" / "figures"
dossier_tableaux = racine / "reports" / "tables"

dossier_figures.mkdir(parents=True, exist_ok=True)
dossier_tableaux.mkdir(parents=True, exist_ok=True)

# Enregistrer le graphique
fig_corr.savefig(
    dossier_figures / "correlations_spearman_44.png",
    dpi=200, bbox_inches="tight",
)

# Conserver les coefficients sans les arrondir
correlations.to_csv(
    dossier_tableaux / "correlations_spearman_44.csv",
    encoding="utf-8-sig",
)

# Conserver le nombre d'observations utilisé pour chaque paire
presence = donnees_correlation.notna().astype(int)
effectifs_correlations = presence.T @ presence

effectifs_correlations.to_csv(
    dossier_tableaux / "effectifs_correlations_44.csv",
    encoding="utf-8-sig",
)

print("Graphique, coefficients et effectifs enregistrés.")